# Win Model Evaluation

Overall metrics (single split + CV), then trajectory analysis showing
how the model's ability to identify the winner evolves episode-by-episode.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from src.models.win import (
    train_eval_pipeline, cross_validate,
    metrics_by_episode_number, winner_rank_detail,
)
from src.features.build import build_modeling_table
from src.load import load_data

In [ ]:
data = load_data()
df = build_modeling_table(data)

## Overall metrics

Single train/test split (seasons 1-40 → 41-49) and expanding-window CV.
These give one number averaging over all episodes equally.

In [ ]:
print("=== Single split ===\n")
results = train_eval_pipeline(df)

print("\n\n=== Cross-validation ===\n")
cv_results = cross_validate(df)

## Accuracy trajectory (cross-validated)

In [ ]:
# Pool predictions from all CV folds (non-overlapping test seasons → ~49 seasons total)
all_cv_preds = pd.concat([fold["predictions"] for fold in cv_results["folds"]])

# Get accuracy metrics at each episode number, averaged across seasons
by_ep_cv = metrics_by_episode_number(all_cv_preds)
by_ep_cv = by_ep_cv[by_ep_cv["n_seasons"] >= 10]  # drop noisy tail episodes

MODEL_COLOR = "#0d9488"
BASELINE_COLOR = "#9ca3af"
marker_sizes = by_ep_cv["n_seasons"] * 4  # bigger dot = more seasons behind it
z95 = 1.96

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Win model accuracy over season (cross-validated)", fontsize=13, y=1.02)

# Left plot: where does the model rank the winner? (lower = better)
ax = axes[0]
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    by_ep_cv["mean_winner_rank"] - z95 * by_ep_cv["se_winner_rank"],
    by_ep_cv["mean_winner_rank"] + z95 * by_ep_cv["se_winner_rank"],
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_baseline_rank"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Mean winner rank (lower = better)")
ax.set_title("Winner rank")
ax.legend()
ax.invert_yaxis()

# Right plot: how often is the winner the model's #1 pick?
ax = axes[1]
ax.plot(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    (by_ep_cv["top1_accuracy"] - z95 * by_ep_cv["se_top1"]).clip(0),
    (by_ep_cv["top1_accuracy"] + z95 * by_ep_cv["se_top1"]).clip(0, 1),
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["baseline_top1"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Top-1 accuracy")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()

fig.tight_layout()
plt.show()

print(by_ep_cv.to_string(index=False))

## Significance test

In [ ]:
# Reuse the per-(season, episode) detail already computed in win.py
detail = winner_rank_detail(all_cv_preds)

# Is the model's rank improvement over random significantly > 0?
t, p = stats.ttest_1samp(detail["rank_improvement"], 0)
print(f"Paired t-test (model rank vs baseline rank, all season-episodes):")
print(f"  Mean improvement: {detail['rank_improvement'].mean():.2f} ranks better than random")
print(f"  t = {t:.2f}, p = {p:.4f}, n = {len(detail)}")